# 01 — Ingesta Bronze: Steam Store API + SteamSpy

## Objetivo
Traer datos crudos de juegos indie desde SteamSpy y la Steam Store API, sin
transformar, y guardarlos como tablas Delta en la capa bronze. El filtrado por
fecha y género y la fusión de ambas fuentes se hacen en `02_silver_cleaning`.

## Tablas que produce
- `bronze_steamspy_indie`: catálogo completo de juegos "Indie" de SteamSpy
- `bronze_store_appdetails`: detalle de la Store API de una muestra aleatoria
  de ese catálogo, con corte al juntar 350 juegos dentro de la ventana
  ene 2023 - dic 2024

## Qué se aprende en este notebook
- Guardar la respuesta cruda como `raw_json` para blindar bronze ante cambios
  de esquema de las APIs
- Muestreo aleatorio con corte temprano en vez de filtrar todo el universo
- Rate limiting con `time.sleep()`
- Escritura idempotente a Delta Lake con `MERGE` por `appid`

In [0]:

import requests as rq
import pandas as pd
import time
import random
import json
from datetime import datetime, timezone
from pprint import pprint
from pyspark import Row
from pyspark.sql import functions as F
from delta.tables import DeltaTable



### Ingesta cruda de SteamSpy (indie)
Se guarda cada registro como `raw_json` (texto sin parsear) en vez de columnas
individuales, para blindar la tabla ante cambios de esquema de SteamSpy.
El parseo a columnas estructuradas se hace en `02_silver_cleaning`.

In [0]:
steamSpy_genre_url = "https://steamspy.com/api.php?request=genre&genre=Indie"

steamSpy_response = rq.get(steamSpy_genre_url)
data = steamSpy_response.json()

records = [
    {"appid": appid, "raw_json": json.dumps(game)}
    for appid, game in data.items()
]

df_bronze = spark.createDataFrame(records).withColumn(
    "ingested_at",
    F.current_timestamp()
)

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_steamspy_indie")

In [0]:
def fetch_store_appdetails(appid, records_games):
    release = None
    try:
        response = rq.get(f"https://store.steampowered.com/api/appdetails?appids={appid}&l=english")
        fetched_at = datetime.now(timezone.utc)
        game_data = response.json()
        game_data = next(iter(game_data.values()))

        records_games.append({
            "appid": appid,
            "fetched_at": fetched_at,
            "success": game_data["success"],
            "raw_json": response.text
        })

        if game_data["success"]:
            try:
                release = datetime.strptime(game_data["data"]["release_date"]["date"], "%b %d, %Y")
            except (ValueError, KeyError):
                pass  # fecha ausente o en formato no estándar — se descarta solo para el conteo

    except Exception as e:
        print(f"Error consultando {appid}: {e}")

    time.sleep(1.5)
    return release

In [0]:
pool = list(data.keys())
random.shuffle(pool)  # se mezcla una sola vez, después se va sacando en orden sin repetir

records_games = []
valid_count = 0
target = 350
batch_size = 20

while valid_count < target and pool:
    batch = [pool.pop() for _ in range(min(batch_size, len(pool)))]

    for appid in batch:
        release = fetch_store_appdetails(appid, records_games)

        if release and datetime(2023, 1, 1) <= release < datetime(2025, 1, 1):
            valid_count += 1

        if valid_count >= target:
            break

    print(f"Progreso: {valid_count}/{target} válidos, {len(records_games)} consultados en total")

print(f"Listo. Válidos: {valid_count}, consultados en total: {len(records_games)}")


In [0]:
df_bronze = (
    spark.createDataFrame(records_games)
    .withColumn("ingested_at", F.current_timestamp())
)

if not spark.catalog.tableExists("bronze_store_appdetails"):
    df_bronze.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("bronze_store_appdetails")
else:
    target_table = DeltaTable.forName(spark, "bronze_store_appdetails")
    (
        target_table.alias("target")
        .merge(
            df_bronze.alias("source"),
            "target.appid = source.appid"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
spark.sql("SELECT success, COUNT(*) FROM bronze_store_appdetails GROUP BY success").show()

## Bronze ingestion completado — 7 de octubre 2026

**Tablas creadas:**
- `bronze_steamspy_indie` — 61,504 registros crudos (catálogo completo de juegos
  etiquetados "Indie" en SteamSpy, sin filtrar), formato `raw_json`
- `bronze_store_appdetails` — 1,473 appids consultados vía Store API (muestreo
  aleatorio del catálogo anterior, con corte al alcanzar 350 válidos dentro de
  la ventana ene 2023 - dic 2024), formato `raw_json`

**Resultado del muestreo:** 350 juegos válidos de 1,473 consultados (~24% de
tasa de acierto). 1,470 consultas exitosas, 3 sin datos (`success: false`).

**Nota:** `bronze_store_appdetails` contiene todos los appids consultados,
no solo los 350 válidos — el filtrado por fecha y género, y la fusión con
los datos de SteamSpy, se hacen en `02_silver_cleaning`.